# AutoVoice: Train DistilBERT Token Classification (NER) on GPU
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/AutoVoice/blob/main/notebooks/02_train_ner_colab.ipynb)

This notebook fine-tunes `distilbert-base-uncased` for sequence token labeling to extract:
- `COMPONENT`: Automotive subsystems and parts (e.g. *steering rack*, *master cylinder*, *airbag*)
- `SYMPTOM`: Failure behaviors and defects (e.g. *stalled*, *leaking fluid*, *smoke*, *locked up*)

**Google Colab Ready:**
- Select **Runtime > Change runtime type > T4 GPU**.
- Prompts to upload `complaints.parquet` if not present.
- When training finishes, automatically zips and downloads `autovoice_ner_model_artifacts.zip` to your computer.


In [ ]:
# 1. Environment Setup & Dependency Installation
!pip install -q transformers==4.40.0 datasets seqeval accelerate torch

import os, sys, json, shutil, re
from pathlib import Path
import pandas as pd
import numpy as np
import torch

try:
    from google.colab import files
    IN_COLAB = True
    print('🚀 Running in Google Colab.')
except ImportError:
    IN_COLAB = False
    print('💻 Running locally.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Hardware Accelerator: {device}')
if device == 'cuda':
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Interactive Data Loading
data_file = None
candidate_paths = [
    Path('complaints.parquet'),
    Path('data/processed/complaints.parquet'),
    Path('../data/processed/complaints.parquet'),
    Path('nhtsa_complaints_raw.csv'),
]

for cp in candidate_paths:
    if cp.exists():
        data_file = cp
        break

if data_file is None and IN_COLAB:
    print('📁 Please upload complaints.parquet or nhtsa_complaints_raw.csv:')
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(('.parquet', '.csv')):
            data_file = Path(fname)
            break

if data_file is None:
    raise FileNotFoundError('No complaint dataset found. Please upload complaints.parquet.')

print(f'✅ Loading dataset from {data_file}')
df = pd.read_parquet(data_file) if str(data_file).endswith('.parquet') else pd.read_csv(data_file)
text_col = 'narrative' if 'narrative' in df.columns else ('cdescr' if 'cdescr' in df.columns else df.columns[0])
texts = df[text_col].dropna().astype(str).tolist()
print(f'Loaded {len(texts)} complaint narratives.')


In [ ]:
# 3. Self-Contained BIO Schema & Weak Labeling Engine
LABELS = ['O', 'B-COMPONENT', 'I-COMPONENT', 'B-SYMPTOM', 'I-SYMPTOM']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

# Curated automotive lexicons for weak silver labeling
COMPONENTS = ['steering', 'power steering', 'brakes', 'brake', 'engine', 'transmission', 'airbag', 'air bag', 'suspension', 'battery', 'accelerator', 'throttle', 'tire', 'seat belt', 'fuel pump', 'alternator']
SYMPTOMS = ['stalled', 'stall', 'shut off', 'died', 'won\'t start', 'failed to start', 'fire', 'caught fire', 'smoke', 'smoking', 'crash', 'collision', 'locked up', 'vibration', 'shaking', 'leaking', 'leak', 'loss of power', 'warning light', 'loss of control']

def tag_text(text):
    words = text.split()
    tags = ['O'] * len(words)
    text_lower = text.lower()
    # Match symptoms
    for sym in SYMPTOMS:
        for m in re.finditer(rf'\b{re.escape(sym)}\b', text_lower):
            start_char = m.start()
            end_char = m.end()
            curr_char = 0
            for i, w in enumerate(words):
                w_start = text.find(w, curr_char)
                w_end = w_start + len(w)
                curr_char = w_end
                if w_start >= start_char and w_end <= end_char:
                    tags[i] = 'B-SYMPTOM' if tags[i] == 'O' and (i == 0 or 'SYMPTOM' not in tags[i-1]) else 'I-SYMPTOM'
    # Match components
    for comp in COMPONENTS:
        for m in re.finditer(rf'\b{re.escape(comp)}\b', text_lower):
            start_char = m.start()
            end_char = m.end()
            curr_char = 0
            for i, w in enumerate(words):
                w_start = text.find(w, curr_char)
                w_end = w_start + len(w)
                curr_char = w_end
                if w_start >= start_char and w_end <= end_char and tags[i] == 'O':
                    tags[i] = 'B-COMPONENT' if (i == 0 or 'COMPONENT' not in tags[i-1]) else 'I-COMPONENT'
    return words, [label2id[t] for t in tags]

sample_words, sample_tags = tag_text('The power steering failed at 65 mph causing loss of control and crash.')
for w, t in zip(sample_words, sample_tags):
    if t != 0:
        print(f'{w} -> {id2label[t]}')


In [ ]:
# 4. Build Dataset & Tokenizer Alignment
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_ID = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

all_tokens, all_ner_tags = [], []
for t in texts:
    words, tags = tag_text(t)
    if words:
        all_tokens.append(words)
        all_ner_tags.append(tags)

raw_dataset = Dataset.from_dict({'tokens': all_tokens, 'ner_tags': all_ner_tags})
split_dataset = raw_dataset.train_test_split(test_size=0.2, seed=42)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True, max_length=256)
    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx] if word_idx < len(label) else -100)
            else:
                # Inside same word sub-token
                cur_label = label[word_idx] if word_idx < len(label) else -100
                label_ids.append(cur_label)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized

train_ds = split_dataset['train'].map(tokenize_and_align_labels, batched=True)
eval_ds = split_dataset['test'].map(tokenize_and_align_labels, batched=True)
print(f'Train examples: {len(train_ds)}, Test examples: {len(eval_ds)}')


In [ ]:
# 5. Fine-Tune DistilBERT with seqeval Evaluation
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate

seqeval_metric = evaluate.load('seqeval')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [id2label[p_val] for (p_val, l_val) in zip(prediction, label) if l_val != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l_val] for (p_val, l_val) in zip(prediction, label) if l_val != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy'],
    }

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir='./results_ner',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

print('🚀 Commencing DistilBERT training...')
trainer.train()


In [ ]:
# 6. Evaluate Model on Test Split
eval_results = trainer.evaluate()
print('Verified NER Evaluation Metrics:', json.dumps(eval_results, indent=2))

export_dir = Path('autovoice_ner_model')
export_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(export_dir)
tokenizer.save_pretrained(export_dir)
(export_dir / 'ner_metrics.json').write_text(json.dumps(eval_results, indent=2))
print('Saved model and metrics to:', export_dir)


In [ ]:
# 7. Interactive Sample Inference
from transformers import pipeline

ner_pipe = pipeline('ner', model=str(export_dir), tokenizer=str(export_dir), aggregation_strategy='simple')
sample_narrative = 'While driving at 65 mph, the power steering suddenly locked up, resulting in brake failure and a crash.'
print('Inference Sample:', sample_narrative)
extracted = ner_pipe(sample_narrative)
for ent in extracted:
    print(f'Entity: {ent["word"]} -> {ent["entity_group"]} (Score: {ent["score"]:.4f})')


In [ ]:
# 8. Automated Model Artifact Packaging & Browser Download
archive_name = 'autovoice_ner_model_artifacts'
shutil.make_archive(archive_name, 'zip', export_dir)
print(f'📦 Packaged DistilBERT NER model artifacts into {archive_name}.zip')

if IN_COLAB:
    print('⬇️ Downloading model artifacts to your local machine...')
    files.download(f'{archive_name}.zip')
else:
    print(f'Archive created at: {Path(archive_name + ".zip").resolve()}')
